In [1]:
# load sample list
import os
import json
import sys
sys.path.append("..")
os.environ["CUDA_VISIBLE_DEVICES"]="0"
from openTSNE import TSNE
import torch

from train_files.train_router_primevul_choice_8models import RouterDataset, RouterModule
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from transformers import AutoTokenizer, DebertaV2Model

dataset_paths = ["../datasets/primevul/balance_8models/choice/train.json"]
data_types    = ["probability"]   # multiple choice → Eq. 2 scores

with open(dataset_paths[0]) as _f:
    number_per_dataset = len(json.load(_f)) 

tokenizer     = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base", truncation_side='left', padding=True)
encoder_model = DebertaV2Model.from_pretrained("microsoft/mdeberta-v3-base").to("cuda")

router_datasets = [RouterDataset(data_path, data_type=data_types[i], dataset_id=i, size=number_per_dataset) for i, data_path in enumerate(dataset_paths)]
for router_dataset in router_datasets:
    router_dataset.register_tokenizer(tokenizer)
router_dataset = ConcatDataset(router_datasets)
router_dataloader = DataLoader(router_dataset, batch_size=64)

router_model = RouterModule(encoder_model, hidden_state_dim=768, node_size=len(router_datasets[0].router_node), similarity_function="cos").to("cuda")

# get embeddings from pre-trained encoder
all_hidden_states = []
with torch.no_grad():
    for i, batch in enumerate(router_dataloader):
        inputs, _, dataset_id, cluster_id = batch
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
        _, hidden_states = router_model(**inputs)
        all_hidden_states.append(hidden_states.cpu())

all_hidden_states = torch.concat(all_hidden_states)
print(f"Embeddings shape: {all_hidden_states.shape}")

/home/minh.le4/RouterDC/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/minh.le4/RouterDC/venv/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Embeddings shape: torch.Size([8405, 768])


In [2]:
from openTSNE import TSNE

np_hidden_states = all_hidden_states.cpu().numpy()
tsne_result = TSNE(n_components=5, n_jobs=12, negative_gradient_method='bh').fit(np_hidden_states)
print(f"t-SNE result shape: {tsne_result.shape}")

/home/minh.le4/RouterDC/venv/lib/python3.10/site-packages/openTSNE/tsne.py:1468: FutureWarning: BH t-SNE for >3 dimensions can lead to segfaults and is generally a bad idea. In the future, calling BH with >3 dimensions will raise a RuntimeError
  warnings.warn(


t-SNE result shape: (8405, 5)


In [3]:
from sklearn.cluster import KMeans
import numpy as np
import random
import json

n_clusters = 5   # paper uses N=5
seed = 42
random.seed(seed)
np.random.seed(seed)

base_output_path = "../datasets/primevul/balance_8models/choice_cluster/"
os.makedirs(base_output_path, exist_ok=True)

x = tsne_result
kmeans = KMeans(n_clusters=n_clusters, max_iter=1000)
kmeans.fit(x)
kmeans_labels = kmeans.labels_.tolist()

# only one dataset so labels_split[0] = all labels
labels_split = [kmeans_labels[i * number_per_dataset: (i + 1) * number_per_dataset] for i in range(len(dataset_paths))]

for i, data_path in enumerate(dataset_paths):
    cluster_ids = labels_split[i]

    with open(data_path, 'r') as f:
        sample_list = json.load(f)

    new_sample_list = []
    for j, sample in enumerate(sample_list):   # no cutoff — use all samples
        new_sample = sample
        new_sample['cluster_id'] = cluster_ids[j]
        new_sample_list.append(new_sample)

    out_name = data_path.split('/')[-1]
    out_path = os.path.join(base_output_path, out_name)
    with open(out_path, 'w') as f:
        json.dump(new_sample_list, f)

    print(f"Saved {len(new_sample_list)} samples to {out_path}")
    print(f"Cluster distribution: { {c: cluster_ids.count(c) for c in sorted(set(cluster_ids))} }")

Saved 8405 samples to ../datasets/primevul/balance_8models/choice_cluster/train.json
Cluster distribution: {0: 724, 1: 1944, 2: 3412, 3: 1405, 4: 920}
